### Трансферное обучение

Изначально была выдвинута идея, что если взять обученную глубокую НС, например, VGG-16 или 19, или любую из сетей ResNet, то ее относительно легко можно переделать для решения схожих задач. Например, сеть VGG на выходе дает 1000 классов, а для решения текущей задачи требуется всего 10 классов. Почему бы тогда просто не заменить полносвязные слои на аналогичные, которые на выходе будут формировать 10 значений вместо 1000?

Трансферное обучение — дообучение полносвязного слоя глубокой нейронной сети для выполнения иных задач

In [22]:
import os
import json
import torch.utils.data as data

from torchvision import models
from PIL import Image

import tqdm as tqdm
import torch.nn as nn
import torchvision.transforms.v2 as tfs_v2
import torch.nn
import torch.optim as optim


class DogDataset(data.Dataset):
    def __init__(self, path, train=True, transform=None):
        self.path = os.path.join(path, "train" if train else "test")
        self.transform = transform
 
        with open(os.path.join(self.path, "format.json"), "r") as fp:
            self.format = json.load(fp)
 
        self.length = 0
        self.files = []
 
        for _dir, _target in self.format.items():
            path = os.path.join(self.path, _dir)
            list_files = os.listdir(path)
            self.length += len(list_files)
            self.files.extend(map(lambda _x: (os.path.join(path, _x), _target), list_files))
 
    def __getitem__(self, item):
        path_file, target = self.files[item]
        img = Image.open(path_file)
        if self.transform:
            img = self.transform(img)
        return img, target 
 
    def __len__(self):
        return self.length

Для трансферного обучения распознавания пород собак будем использовать модель ResNet50

In [23]:
resnet = models.ResNet50_Weights.DEFAULT
transforms = resnet.transforms()

model = models.resnet50(weights=resnet)
model.requires_grad_(False) # Отключаем обучение модели

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Con

Далее, необходимо выделить полносвязный слой fc

In [24]:
model.fc = nn.Linear(512*4, 10)
model.fc.requires_grad_(True)
print(model.parameters)

<bound method Module.parameters of ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsa

In [1]:
d_train = DogDataset(r'datasets/cnn/dogs', transform=transforms)
train_data = data.DataLoader(d_train, batch_size=32, shuffle=True)

optimizer = optim.Adam(params=model.fc.parameters(), lr=0.001, weight_decay=0.001)
loss_function = nn.CrossEntropyLoss()
epochs = 3
model.train()

for e in range(epochs):
    loss_mean = 0
    lm_count  = 0
    
    train_tqdm = tqdm.tqdm(train_data, leave=True)
    for x_train, y_train in train_tqdm:
        predict = model(x_train)
        loss = loss_function(predict, y_train)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        lm_count += 1
        loss_mean = 1/lm_count * loss.item() + (1 - 1/lm_count) * loss_mean
        train_tqdm.set_description(f'Epoch {e+1}/{epochs}, loss_mean{loss_mean:.3f}')

st = model.state_dict()
torch.save(st, 'model_transfer_resnet.tar')

NameError: name 'DogDataset' is not defined